# ONNX EMBEDDER

source: https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/lessons/09-onnx-embedder.md


We use ONNX to reduce Run Time, where:

sentence-transformers: 4.8 GB, 58 packages

ONNX Runtime: 147 MB, 27 packages

In [6]:
from pathlib import Path
from rich import print
import os
import sys
import numpy as np



parent_directory = os.path.dirname(os.getcwd())
embed_directory = os.path.join(parent_directory,  "embed")
sys.path.append(embed_directory)
# Construct the absolute path to your model files
model_path = os.path.join(parent_directory, "models", "Xenova", "all-MiniLM-L6-v2")
from embedder import Embedder
# Pass it explicitly to the class initialization
embed = Embedder(path=model_path)


In [7]:
q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

# compute similarity
print(f'the similarity between q1 and d is: {v1.dot(dv)}')

print(f'the similarity between q2 and d is: {v2.dot(dv)}')

the similarity between q1 and d is: 0.32332403170761753

the similarity between q2 and d is: 0.01973043944798833

### use onnx as embedder

In [ ]:
from ingest import load_faq_data, build_index
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document



# 1. load the documents & extract only useful fields
print('---------------------------------------------------------------')
print('Step 1: load the data and get question and answer fields only')
documents = load_faq_data()
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

print(f'number of text is: {len(texts)}')
print(f'text example is: {texts[0]}')

---------------------------------------------------------------

Step 1: load the data and get question and answer fields only

number of text is: 1350

text example is: Course: When does the course start? A new cohort runs roughly January–April every year. For the 
current cohort's exact start date and registration link, check the 
(https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the (https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` 
channel.

In [ ]:
# 2. convert the list of text into documents
langchain_docs = [Document(page_content=text) for text in texts]


# Note::: I changed the chunck_size to 5000 so that I get the same numbers of vectors and documents to overcome an error
# 3. Setup the Recursive Text Splitter
# It chunks your text smartly, keeping paragraphs/sentences together, and creates overlaps!
# NOTE::: the number of vectors need to be exactly as the number of documents if I use minsearch
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,    # A high number ensures a single Q&A text never gets split up
    # chunk_size=500, # for improvement
    chunk_overlap=0,    # Zero overlap ensures no repeated data, just like your old loop
    # chunk_overlap=50, # for improvement
    # length_function=lambda text: len(tokenizer.encode(text)) # Counting tokens, # for improvement and by default length_function=len
    # separators=[r"Question \d+:", "\n\n", "\n", " "],# for improvement. note custome seperator= separators=["\n===\n", "\n\n", "\n", " "] but since I have only question and answer, I set it differently
    # is_separator_regex=True # for improvement 
)

# split the docs (chuncks)
texts_chuncks = text_splitter.split_documents(langchain_docs) # Note: we do not use  text_splitter.split_text(" ".join(texts))  since we have a list of document, each document will be splitted into chuncks, then langchain merges all the chunck

print(f"The number of chunks documents is: {len(texts_chuncks)}")
# print(texts_chuncks[0])

The number of chunks documents is: 1350

In [14]:
print(texts_chuncks[0])

Document(
    metadata={},
    page_content="Course: When does the course start? A new cohort runs roughly January–April every year. For the 
current cohort's exact start date and registration link, check the [course repo 
README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo 
before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join
DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."
)

In [20]:
#4. embed the vectores
X= list()
for doc in  texts_chuncks:
    vectors = embed.encode(doc.page_content)
    X.append(vectors)

X = np.array(X)

In [21]:
# compute similaruty
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}